## Experiment Analysis: Number of Images

In [4]:
# Library imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
from scipy.stats import wilcoxon
from scipy import stats

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display the dataframe to fit nicely on the screen
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

#### 1. Data Loading, Processing and Save

Loading the experiment-generated files and converting them into a structured DataFrame for analysis.

In [5]:
# Define the directory containing experiment results
dir_experiments = "../experiments/nimages/results"

# Dictionary to store result file paths for each superpixel/nimage combination
results_files = defaultdict(dict)

# Iterate through each superpixel folder in the results directory
for superpixel_folder in sorted(os.listdir(dir_experiments)):
    superpixel_path = os.path.join(dir_experiments, superpixel_folder)
    # Check if the folder is a valid superpixel folder
    if os.path.isdir(superpixel_path) and superpixel_folder.startswith("super"):
        # Store result files (.csv) found in the folder
        for file in sorted(os.listdir(superpixel_path)):
            if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[superpixel_folder][file] = os.path.join(superpixel_path, file)

# List to store extracted results and nimage values
results_data = []
nimages_values = [1]  # Start with 1 image as default

# Extract metrics from each result file
for folder, contents in results_files.items():
    # Parse superpixel and nimage values from folder name
    superpixel = int(folder.split("_")[0].replace('super', ''))
    nimage = int(folder.split("_")[1].replace('images', ''))
    nimages_values.append(nimage)
    for seed, file in contents.items():
        if isinstance(file, str) and file.endswith('_results.csv'):
            results_file = file
            seed_value = int(seed.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    # Extract accuracy and kappa metrics from the first two lines
                    class_accuracies = list(map(float, lines[0].strip().split(';')[:7]))
                    class1_accuracy, class2_accuracy = class_accuracies[0], class_accuracies[1]
                    class3_accuracy, class4_accuracy = class_accuracies[2], class_accuracies[3]
                    class5_accuracy, class6_accuracy = class_accuracies[4], class_accuracies[5]
                    class7_accuracy = class_accuracies[6]
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    # Append the extracted metrics to the results list
                    results_data.append({
                        'superpixel': superpixel,
                        'nimage': nimage,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'class3_accuracy': class3_accuracy,
                        'class4_accuracy': class4_accuracy,
                        'class5_accuracy': class5_accuracy,
                        'class6_accuracy': class6_accuracy,
                        'class7_accuracy': class7_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })

# Convert the results list to a DataFrame
df_nimage = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file for further analysis
df_nimage.to_csv('nimages_results_summary.csv', index=False)

#### 2. Analyses and Visualization

In [6]:
# Load the DataFrame with superpixel results
df_nsuper = pd.read_csv('nsuperpixels_results_summary.csv')

# Filter results for superpixel=50 only
df_super25 = df_nsuper[df_nsuper['superpixel'] == 25].copy()

# Calculate mean, standard deviation, and nfeat for superpixel=25
metrics = ['class1_accuracy', 'class2_accuracy', 'class3_accuracy', 'class4_accuracy', 'class5_accuracy', 'class6_accuracy', 'class7_accuracy', 'kappa', 'global_accuracy']
mean_super25 = df_super25[metrics].mean()
std_super25 = df_super25[metrics].std()
nfeat_super25 = int(df_super25['nfeat'].mean())

# Make a copy of df_nimage and group results by nimage, calculating mean and std for each metric
df_nimage_copy = df_nimage.copy()
metrics = ['class1_accuracy', 'class2_accuracy', 'class3_accuracy', 'class4_accuracy', 'class5_accuracy', 'class6_accuracy', 'class7_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Group by number of images and calculate mean and std for each metric (except nfeat)
df_nimage_copy = df_nimage_copy.groupby('nimage')[metrics[:-1]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
df_nimage_copy.columns = ['_'.join(col).strip() if col[1] else col[0] for col in df_nimage_copy.columns.values]

# Add the mode of nfeat for each nimage to df_nimage_copy
nfeat_mode = df_nimage.groupby('nimage')['nfeat'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
nfeat_mode = nfeat_mode.sort_index().reset_index()
nfeat_mode.columns = ['nimage', 'nfeat']

df_nimage_copy = df_nimage_copy.merge(nfeat_mode, on='nimage', how='left')

# Add a row for superpixel=50 with the calculated values and nimage=1
df_nimage_copy = pd.concat([df_nimage_copy, pd.DataFrame({
    'nimage': [1],
    'class1_accuracy_mean': [mean_super25['class1_accuracy']],
    'class1_accuracy_std': [std_super25['class1_accuracy']],
    'class2_accuracy_mean': [mean_super25['class2_accuracy']],
    'class2_accuracy_std': [std_super25['class2_accuracy']],
    'class3_accuracy_mean': [mean_super25['class3_accuracy']],
    'class3_accuracy_std': [std_super25['class3_accuracy']],
    'class4_accuracy_mean': [mean_super25['class4_accuracy']],
    'class4_accuracy_std': [std_super25['class4_accuracy']],
    'class5_accuracy_mean': [mean_super25['class5_accuracy']],
    'class5_accuracy_std': [std_super25['class5_accuracy']],
    'class6_accuracy_mean': [mean_super25['class6_accuracy']],
    'class6_accuracy_std': [std_super25['class6_accuracy']],
    'class7_accuracy_mean': [mean_super25['class7_accuracy']],
    'class7_accuracy_std': [std_super25['class7_accuracy']],
    'kappa_mean': [mean_super25['kappa']],
    'kappa_std': [std_super25['kappa']],
    'global_accuracy_mean': [mean_super25['global_accuracy']],
    'global_accuracy_std': [std_super25['global_accuracy']],
    'nfeat': [nfeat_super25]
})], ignore_index=True)

# Sort the DataFrame by number of images
df_nimage_copy = df_nimage_copy.sort_values(by='nimage').reset_index(drop=True)

# Highlight the best metrics in the summary table
summary_stats_nimage = df_nimage_copy[['nimage', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std', 'nfeat']].style.highlight_max(
    subset=['kappa_mean', 'global_accuracy_mean'], color='gray'
)

# Format values as percentages for better presentation
summary_stats_nimage = summary_stats_nimage.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

summary_stats_nimage

,nimage,kappa_mean,kappa_std,global_accuracy_mean,global_accuracy_std,nfeat
0,1,81.5079%,0.015061,88.7693%,0.009316,18150
1,2,83.1803%,0.010224,89.7971%,0.006285,36300
2,3,84.2671%,0.008658,90.4806%,0.005118,54450
3,4,84.8291%,0.009572,90.8086%,0.005939,72600
4,5,85.4358%,0.010998,91.1743%,0.006617,90750


In [7]:
wilcoxon_results = pd.DataFrame(columns=['nimage1', 'nimage2', 'statistic', 'p_value'])

nimages_unique = sorted(df_nimage['nimage'].unique())
for i in range(len(nimages_unique)):
    for j in range(i + 1, len(nimages_unique)):
        n1 = nimages_unique[i]
        n2 = nimages_unique[j]
        
        data_n1 = df_nimage[df_nimage['nimage'] == n1]['global_accuracy']
        data_n2 = df_nimage[df_nimage['nimage'] == n2]['global_accuracy']
        
        if len(data_n1) == len(data_n2) and len(data_n1) > 0:
            statistic, p_value = wilcoxon(data_n1, data_n2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'nimage1': n1,
                'nimage2': n2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Resultados do teste de Wilcoxon entre nimages:")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparações com diferença estatisticamente significativa:")
print(significant)
print(f"Total de comparações significativas: {len(significant)} de {len(wilcoxon_results)}")


Resultados do teste de Wilcoxon entre nimages:
   nimage1  nimage2  statistic   p_value
0        3        4       16.0  0.275391
1        3        5        5.0  0.019531
2        4        5       12.0  0.130859
Comparações com diferença estatisticamente significativa:
   nimage1  nimage2  statistic   p_value
1        3        5        5.0  0.019531
Total de comparações significativas: 1 de 3
